# ValPAS prototype

## Dependencies
Dependencies to run this notebook should be installed according to the README.md located at the root folder of the project.

## Jupyter notebook "frontmatter"
The following lines of code are overhead to make loading of the valpas package possible from within the notebook without having to install the package via pip / setuptools. Essentially this loads the 'src/' folder into the sys path as searchable for modules. 

In [1]:
# iPython magic to autoreload modules everytime code is executed to propagate changes to the code
%load_ext autoreload
%autoreload 2

# loading '/src' into sys path
import os
import sys
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

# don't show warnings
import warnings
warnings.filterwarnings('ignore')

### Defining the file path to the sample files:

In [2]:
# path to the csv containing the proteome data
fpath_prot = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "rhodo_multi_proteomics.csv"
    )

# path to the csv containing the metabolomics data
fpath_metabol = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "rhodo_multi_metabolomics.csv"
    )

### Correlation between samples
Correlation can be calculated between instances within one type of data (e.g. protein-protein) or between two types of data (e.g. protein-metabolite). Below both variants are demonstrated (first within one type then across types). The returned data is a series of correlated instances sorted by the strenght of their correlation.

The ValPAS module offers a command line tool to perform all operations (see also `src/valpas/valpas.py`). We will be importing the `main` function from `valpas/valpas.py` and use this to execute commands in this Jupyter Notebook.

In [3]:
from valpas.valpas import main as valpas_main

Next we will calculate the correlation between instances within one data type. In this case the data type are protein abundances. The returned list is sorted descending (i.e. stronges correlation first).

The command issued below is equivalent to calling `python valpas.py associate -i INFILE` from the command line, where `INFILE='../sample_data/rhodo_multi_metabolomics.csv'` as defined in [Defining the file path...](#defining-the-file-path-to-the-sample-files).

If not further defined the script defaults to print the sorted list of id pairs with their association score to `stdout`.

In [4]:
%%capture out --no-stderr
# above line does some magic to capture the stdout
valpas_main(['associate', '-i', fpath_prot])

In [5]:
# some short code to display the first 10 entries printed to stdout 
count = 0
for line in out.stdout.split('\n'):
    print(line)
    count += 1
    if count > 10:
        break

RTO4_ID_1,RTO4_ID_2,Correlation
11652,15738,1.0000000000000004
9095,13762,1.0000000000000004
10605,16024,1.0000000000000004
10703,12656,1.0000000000000004
11694,12317,1.0000000000000004
12550,15105,1.0000000000000004
15720,16109,1.0000000000000004
12817,16620,1.0000000000000004
13879,16377,1.0000000000000004
11467,13762,1.0000000000000004


Next we calculate the correlation between instances across two different types of data (in this example proteins and metabolites). Note that the function `calc_correlation` also accepts keyword parameters. If `fpath_2` is omitted as in the example above only the correlation between instances in the dataset designated by `fpath_1` is calculated. If both `fpath_1` and `fpath2` are present correlation between pairs of instances across both datasets are calculated. Additionally, note that the correlation function `corr_func` can be defined. Options are `'pearson'`, `'spearman'` and `'kendall'`. Note that if omitted the calculation defaults to `'pearson'`.

We can also further define an output path for the generated csv. In this case we will store the data to `'../sample_data/rhodo_multi_prot-metabol_corr.csv'` (defined by the parameter `-o OUTFILE`).

Again the equivalent command on the command line would be `python valpas.py associate -i INFILE -I INFILE2 -a spearman -o OUTFILE` where `INFILE='../sample_data/rhodo_multi_proteomics.csv'` and `INFILE2='../sample_data/rhodo_multi_metabolomics.csv'` as defined in [Defining the file path...](#defining-the-file-path-to-the-sample-files).



In [6]:
# defining the outfile path
out_fpath = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "rhodo_multi_prot-metabol_corr_list.csv"
    )

valpas_main([
    'associate',
    '-i', fpath_prot,
    '-I', fpath_metabol,
    '-a', 'spearman',
    '-o', out_fpath
    ])

It is also possible to print the resulting correlation matrix raw instead of outputting a sorted list of id pairs with their association score. This can be achieved using the `'-ot correlation_matrix'` option of the command line tool. Note if `'-ot'` is omitted it defaults to `'sorted_list'` and prints / stores said list. A example command is shown below.

The resulting CSV contains ids in the header as well as first column

In [9]:
out_fpath = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "rhodo_multi_metabol-metabol_corr_mat.csv"
    )

valpas_main([
    'associate',
    '-i', fpath_metabol,
    '-o', out_fpath,
    '-ot', 'correlation_matrix'
    ])